# Численные эксперименты по применению методов обучения с подкреплением к задаче коммивояжера

Комплекс численных экспериментов для исследования нейросетевых методов решения задачи коммивояжера (TSP). Включает: задание управляющих параметров, генерацию конфигураций, оптимизацию параметров моделей (обучение), проведение пяти тестов и статистический анализ результатов.

Объекты исследования (тесты):
1. Зависимость точности и времени расчета от размерности задачи (масштабируемость).
2. Независимость качества решений относительно перенумерации вершин графа.
3. Устойчивость моделей к возмущениям входных координат.
4. Устойчивость оптимизации к случайным погрешностям вычисления целевой функции.
5. Корреляция внутренних вероятностных метрик алгоритма с погрешностью найденного маршрута.

## 0. Настройка параметров эксперимента

Задание режима вычислений и директории сохранения результатов.

`SCALE = "small"` - отладочный запуск на задачах минимальной размерности для проверки корректности программного кода.

`SCALE = "full"` - основной расчет для конечных результатов.

Логические флаги `RUN_*` задают выборочный запуск этапов для исключения повторных вычислений. Каждому циклу присваивается идентификатор RUN_ID. Вычисления привязаны к текущему интерпретатору Python.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import Image, display
import sys

PYTHON = sys.executable
PROJECT_ROOT = Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
existing_pythonpath = os.environ.get("PYTHONPATH")
os.environ["PYTHONPATH"] = str(SRC_PATH) if not existing_pythonpath else str(SRC_PATH) + os.pathsep + existing_pythonpath
os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("PYTHONDONTWRITEBYTECODE", "1")
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".runtime-cache" / "matplotlib"))
os.environ.setdefault("RL4TSP_TORCH_THREADS", "4")
SCALE = "small"  # заменить на "full" для итогового запуска
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = PROJECT_ROOT / "results" / RUN_ID
CONFIG_DIR = RUN_ROOT / "configs"

RUN_TRAIN_REINFORCE = True
RUN_TRAIN_POMO = True
RUN_SCALING = True
RUN_DIAGNOSTICS = True
RUN_REWARD_NOISE = True

print(f"Каталог проекта: {PROJECT_ROOT}")
print(f"Режим: {SCALE}")
print(f"Результаты: {RUN_ROOT}")

## 1. Формирование конфигураций

Генерация конфигурационных файлов на основе шаблонов с динамической подстановкой путей к результатам и сохраненным параметрам моделей. Обеспечивает воспроизводимость расчетов за счет фиксации параметров в папке `results/<RUN_ID>/configs/`.

In [ ]:
def read_config(name: str) -> dict:
    return json.loads((PROJECT_ROOT / "configs" / name).read_text(encoding="utf-8"))


def write_config(name: str, config: dict) -> Path:
    CONFIG_DIR.mkdir(parents=True, exist_ok=True)
    path = CONFIG_DIR / name
    path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
    return path


def reinforce_checkpoint_entries(output_root: Path) -> list[dict]:
    return [
        {"method": "Attention:batch_greedy_mean", "model": "attention", "decoder": "greedy", "checkpoint": str(output_root / "models" / "attention_batch_greedy_mean.pt")},
        {"method": "Attention:greedy_rollout", "model": "attention", "decoder": "greedy", "checkpoint": str(output_root / "models" / "attention_greedy_rollout.pt")},
        {"method": "PointerNet:batch_greedy_mean", "model": "pointer", "decoder": "greedy", "checkpoint": str(output_root / "models" / "pointer_batch_greedy_mean.pt")},
        {"method": "PointerNet:greedy_rollout", "model": "pointer", "decoder": "greedy", "checkpoint": str(output_root / "models" / "pointer_greedy_rollout.pt")},
    ]


def pomo_checkpoint_entry(output_root: Path) -> dict:
    return {"method": "POMO", "model": "attention", "decoder": "pomo", "checkpoint": str(output_root / "models" / "attention_pomo.pt"), "pomo_num_starts": None}


def prepare_configs(scale: str) -> dict[str, Path]:
    if scale not in {"small", "full"}:
        raise ValueError("scale должен быть 'small' или 'full'")
    if RUN_ROOT.exists():
        raise FileExistsError(f"Каталог запуска уже существует: {RUN_ROOT}")
    suffix = "small" if scale == "small" else "full"
    output_root = RUN_ROOT / scale
    paths = {}

    reinforce = read_config(f"tsp_reinforce_{suffix}.json")
    reinforce["output_root"] = str(output_root)
    paths["reinforce"] = write_config("tsp_reinforce.json", reinforce)

    pomo = read_config(f"tsp_pomo_{suffix}.json")
    pomo["output_root"] = str(output_root)
    paths["pomo"] = write_config("tsp_pomo.json", pomo)

    reinforce_entries = reinforce_checkpoint_entries(output_root)
    pomo_entry = pomo_checkpoint_entry(output_root)

    compare = read_config(f"tsp_compare_{suffix}.json")
    compare["output_root"] = str(output_root)
    compare["model_checkpoints"] = reinforce_entries
    compare["pomo_checkpoint"] = str(output_root / "models" / "attention_pomo.pt")
    paths["compare"] = write_config("tsp_compare.json", compare)

    diagnostics = read_config(f"tsp_diagnostics_{suffix}.json")
    diagnostics["output_root"] = str(output_root)
    diagnostics["model_checkpoints"] = reinforce_entries + [pomo_entry]
    diagnostics["entropy_model_checkpoints"] = reinforce_entries + [pomo_entry]
    paths["diagnostics"] = write_config("tsp_diagnostics.json", diagnostics)

    reward_noise = read_config(f"tsp_reward_noise_{suffix}.json")
    reward_noise["output_root"] = str(output_root)
    paths["reward_noise"] = write_config("tsp_reward_noise.json", reward_noise)
    return paths


CONFIG_PATHS = prepare_configs(SCALE)
CONFIG_PATHS

## 2. Оптимизация параметров моделей (обучение)

Подготовительный этап для получения расчетных моделей (стратегий).
Оптимизируются параметры четырех вариантов алгоритма REINFORCE (архитектуры Attention и PointerNet с двумя методами снижения дисперсии градиента) и модели POMO со множественным стартом. Итоговым результатом является состояние модели, показавшее наилучшее значение целевой функции в процессе оптимизации.

In [ ]:
def run_step(name: str, *args: str) -> None:
    print(f"\n=== {name} ===", flush=True)
    subprocess.run(args, cwd=PROJECT_ROOT, text=True, check=True, env=os.environ.copy())


if RUN_TRAIN_REINFORCE:
    run_step("REINFORCE: Attention и PointerNet", PYTHON, "experiments/run_tsp_reinforce.py", "--config", str(CONFIG_PATHS["reinforce"]))

if RUN_TRAIN_POMO:
    run_step("POMO", PYTHON, "experiments/run_tsp_pomo.py", "--config", str(CONFIG_PATHS["pomo"]))

## 3. Опыт 1: Зависимость качества решений от размерности задачи

**Цель:** Оценка точности и вычислительной трудоемкости алгоритмов при увеличении числа вершин графа. Модели оптимизированы на размерности $n = 20$, тестируются на размерностях от 10 до 300.

**Методика:** Для каждой размерности генерируется выборка случайных евклидовых графов. Вычисляются: длина маршрута, относительная погрешность и время расчета. Эталоны: точный метод Хелда-Карпа (для $n \le 10$) и эвристика Лина-Кернигана LKH (для $n > 10$).

**Анализ результатов:** Рост относительной погрешности при увеличении числа вершин определяет границы способностей моделей к экстраполяции. Классические методы используются в качестве объективного эталона.

In [ ]:
if RUN_SCALING:
    run_step("Сравнение качества и времени", PYTHON, "experiments/run_tsp_compare.py", "--config", str(CONFIG_PATHS["compare"]))
    summary_path = RUN_ROOT / SCALE / "tsp_comparison" / "summary.csv"
    run_step("Графики масштабирования", PYTHON, "experiments/plot_tsp_summary.py", "--summary", str(summary_path), "--output-dir", str(RUN_ROOT / SCALE / "figures"))

scaling_summary = pd.read_csv(RUN_ROOT / SCALE / "tsp_comparison" / "summary.csv")
display(scaling_summary)
for image_path in [RUN_ROOT / SCALE / "figures" / "tsp_gap.png", RUN_ROOT / SCALE / "figures" / "tsp_time.png"]:
    display(Image(filename=str(image_path)))

## 4. Опыт 2: Инвариантность решения относительно перенумерации вершин графа

**Цель:** Проверка независимости решений от индексации вершин графа.

**Методика:** Сравнение маршрутов для исходного графа и для графа со случайной перестановкой индексов вершин. Метрики: среднее расстояние Хемминга для канонических форм маршрутов, разность длин путей, частота полного совпадения.

**Анализ результатов:** Отклонение метрики Хемминга от нуля указывает на чувствительность архитектуры нейросети или процедуры декодирования к порядку подачи данных.

In [ ]:
if RUN_DIAGNOSTICS:
    run_step("Диагностики TSP", PYTHON, "experiments/run_tsp_diagnostics.py", "--config", str(CONFIG_PATHS["diagnostics"]))

permutation_summary = pd.read_csv(RUN_ROOT / SCALE / "tsp_diagnostics" / "permutation_invariance_summary.csv")
display(permutation_summary)
display(Image(filename=str(RUN_ROOT / SCALE / "figures" / "permutation_invariance.png")))

## 5. Опыт 3: Устойчивость алгоритма к возмущениям входных данных

**Цель:** Оценка чувствительности оптимизированных моделей к погрешности координат вершин.

**Методика:** К координатам вершин тестовых графов добавляется гауссовский шум со стандартным отклонением $\sigma$. Маршрут строится по зашумленным координатам, но его длина оценивается по исходной (невозмущенной) геометрии.

**Анализ результатов:** Оценивается зависимость фактической потери качества решения от величины $\sigma$. Контрольная точка — нулевой уровень шума.

In [ ]:
noise_summary = pd.read_csv(RUN_ROOT / SCALE / "tsp_diagnostics" / "noise_robustness_summary.csv")
display(noise_summary)
display(Image(filename=str(RUN_ROOT / SCALE / "figures" / "noise_robustness.png")))

## 6. Опыт 4: Устойчивость процесса оптимизации к случайным погрешностям целевой функции

**Цель:** Оценка сходимости градиентного поиска при наличии случайной компоненты в наблюдаемом значении функционала качества.

**Методика:** На этапе оптимизации параметров к вычисленной длине маршрута добавляется относительный гауссовский шум $\sigma_{rel}$. Полученные модели тестируются на фиксированной контрольной выборке без шума.

**Анализ результатов:** Сравнение финальной точности моделей, оптимизированных при различных уровнях $\sigma_{rel}$. Значение $\sigma_{rel} = 0.0$ является контрольной группой.

In [ ]:
if RUN_REWARD_NOISE:
    run_step("Reward-noise", PYTHON, "experiments/run_tsp_reward_noise.py", "--config", str(CONFIG_PATHS["reward_noise"]))

reward_summary = pd.read_csv(RUN_ROOT / SCALE / "tsp_reward_noise" / "summary.csv")
display(reward_summary)
for image_path in [RUN_ROOT / SCALE / "figures" / "reward_noise_gap.png", RUN_ROOT / SCALE / "figures" / "reward_noise_training.png"]:
    display(Image(filename=str(image_path)))

## 7. Опыт 5: Анализ взаимосвязи внутренней неопределенности алгоритма и погрешности решения

**Цель:** Проверка возможности использования внутренних вероятностных параметров модели для апостериорной оценки качества решения.

**Методика:** Вычисление коэффициента линейной корреляции Пирсона $r$ между относительной погрешностью маршрута и тремя параметрами: средней энтропией распределения вероятностей, контрастностью (разностью логарифмов вероятностей двух лучших действий) и дисперсией длины при многократном стохастическом поиске на одном графе.

**Анализ результатов:** Высокое абсолютное значение $r$ позволяет использовать внутренние метрики для самодиагностики точности. Низкое значение означает отсутствие связи.

In [ ]:
entropy_summary = pd.read_csv(RUN_ROOT / SCALE / "tsp_diagnostics" / "entropy_experiment_summary.csv")
display(entropy_summary)
for image_path in [RUN_ROOT / SCALE / "figures" / "entropy_gap.png", RUN_ROOT / SCALE / "figures" / "entropy_correlations.png"]:
    display(Image(filename=str(image_path)))

## 8. Обоснование выбора гиперпараметров для полноценного запуска

Результаты малого запуска (`*_small.json`) используются только для проверки работоспособности кода. Итоговые выводы базируются на основном расчете (`full`).

### Общие параметры

- **Фиксация случайных последовательностей:** `seed = 42/43/44/46`для разграничения выборок между этапами обучения и тестирования и обеспечения воспроизводимости.
- **Масштаб области:** Координаты генерируются в квадрате `[0, 10]^2`. Постоянное масштабирование не меняет оптимальный маршрут, но дает длины маршрутов порядка десятков, что удобно для численной стабильности REINFORCE и чтения таблиц.
- **Вычислительное устройство:** device = cpu (фиксированный режим вычислений).
- **Параметры моделей:** Скрытая размерность векторов состояния `hidden_dim = 128`. Число слоев в модели Attention — 3.
- **Параметры оптимизации:** Размерность графов $n = 20$, `epochs = 3000`, шаг градиентного спуска `learning_rate = 0.0001`, ограничение нормы градиента — 1.0, размер подвыборки `batch_size = 64`.
- **Тестовые параметры:** Сетка размерностей `sizes = [10, 20, 50, 100, 200, 300]`. Объем выборок — 300 графов на размерность (500 для тестов с шумом).
- **Ограничение точного расчета:** Граница применения метода Хелда-Карпа — `exact_max_n = 10` (ввиду трудоемкости $O(n^2 2^n)$). Для больших размерностей применяется эвристика LKH с масштабным коэффициентом `lkh_scale = 1000000`.


## 9. Статистическая достоверность результатов: обоснование и ограничения

На графиках приводятся выборочные средние значения и 95%-е доверительные интервалы для математического ожидания, рассчитанные по распределению Стьюдента:$$t_{0.975, n-1} \cdot \frac{s}{\sqrt{n}}$$

где $s$ — выборочное стандартное отклонение, $n$ — объем выборки. Для коэффициентов корреляции Пирсона применяется преобразование Фишера.

**Ограничения анализа:**

1. Локальность: Статистические оценки справедливы только для распределения случайных евклидовых графов в ограниченных областях.

1. **Характер сравнения:** Приведенные доверительные интервалы являются маргинальными. Для строгой проверки гипотез о различиях алгоритмов вычисления организованы на связанных выборках (на одних и тех же графах), что требует применения парных критериев.

1. **Единичный запуск оптимизации:** В эксперименте по зашумлению целевой функции зафиксирована одна траектория обучения. Доверительные интервалы отражают только разброс качества на тестовой выборке графов.

## 10. Основные положения

1. Реализован детерминированный комплекс численных экспериментов по сопоставлению нейросетевых алгоритмов (Attention, PointerNet, POMO) с точным методом Хелда-Карпа и эвристикой LKH на идентичных выборках случайных графов.

1. Экспериментально доказано отсутствие сильной линейной корреляции между внутренней неопределенностью вероятностной модели (энтропией, дисперсией стохастического поиска) и фактической погрешностью маршрута при выходе за пределы размерности обучения. Внутренние метрики уверенности модели не являются надежным индикатором точности решения.

## 11. Заключительный анализ

**Предмет исследования:** Конкретные модификации нейросетевых архитектур, оптимизированные методом REINFORCE на евклидовых задачах размерности n=20. Цель работы — определение математических свойств моделей (масштабируемость, инвариантность, робастность), а не превосходство над методом LKH.

**Достоверность численного контура:** Воспроизводимость обеспечивается фиксацией конфигураций и разделением этапов вычислений. Использование эвристики LKH в качестве численного референса исключает методологическую ошибку сравнения со слабым эталоном.

**Ограничения интерпретации:** Выводы ограничены исследуемыми архитектурами и евклидовой постановкой задачи. Выбор наилучшего состояния модели производился по обучающей выборке. Использование коэффициента Пирсона фиксирует только линейные компоненты взаимосвязи метрик.